# HK BVAR Supplementary Notebook


## Setup


In [ ]:
# NumPy/Alexandria compatibility.
import numpy as _np_tmp
if not hasattr(_np_tmp, 'float_'):
    _np_tmp.float_ = _np_tmp.float64
if not hasattr(_np_tmp, 'complex_'):
    _np_tmp.complex_ = _np_tmp.complex128
if not hasattr(_np_tmp, 'int_'):
    _np_tmp.int_ = _np_tmp.int64
del _np_tmp

import warnings
warnings.filterwarnings('ignore')

import time
from pathlib import Path

import numpy as np
np.random.seed(67)
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from alexandria import MinnesotaBayesianVar
import alexandria.vector_autoregression.var_utilities as vu


In [2]:
# Configuration
DATA = 'data/hk_macro_varx_ready.csv'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

EXOG = ['us_ffr', 'china_gdp']
LAGS = 4
PI1, PI2, PI3 = 0.085, 1.0, 1.0
PI4 = 100.0

# Order used in early baseline checks.
ENDOG_OLD = [
    'hk_exports_china_yoy', 'gdp_growth', 'cpi_inflation',
    'unemployment', 'hibor_3m', 'hk_property_price_qoq'
]
AR_COEF_OLD = [0.627, 0.545, 0.735, 0.991, 0.442, 0.418]

# Canonical order used in robustness checks.
ENDOG_CANONICAL = [
    'hibor_3m', 'hk_exports_china_yoy', 'hk_property_price_qoq',
    'gdp_growth', 'cpi_inflation', 'unemployment'
]
AR_COEF_CANONICAL = [0.442, 0.627, 0.418, 0.545, 0.735, 0.991]

# Aliases for canonical checks.
ENDOG = ENDOG_CANONICAL
AR_COEF = AR_COEF_CANONICAL

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
SAMPLE_LABEL = f"{df.index[0].to_period('Q')}–{df.index[-1].to_period('Q')} | {len(df)} quarters"

print(f'Loaded {DATA}: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Sample: {SAMPLE_LABEL}')
print(f'Legacy order:    {ENDOG_OLD}')
print(f'Canonical order: {ENDOG_CANONICAL}')


Loaded data/hk_macro_varx_ready.csv: 113 rows, 9 columns
Sample: 1998Q1–2026Q1 | 113 quarters
Legacy order:    ['hk_exports_china_yoy', 'gdp_growth', 'cpi_inflation', 'unemployment', 'hibor_3m', 'hk_property_price_qoq']
Canonical order: ['hibor_3m', 'hk_exports_china_yoy', 'hk_property_price_qoq', 'gdp_growth', 'cpi_inflation', 'unemployment']


## Baseline models


In [3]:
# VARX(1) baseline

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
exog   = df[EXOG]
endog3 = df[ENDOG_OLD]
model3 = VAR(endog3, exog=exog)
v1     = model3.fit(maxlags=1)

print(f'VARX(1) | obs={v1.nobs} | AIC={v1.aic:.4f} | BIC={v1.bic:.4f}')
print()

# Ljung-Box
print('Ljung-Box (lag=8):')
for col in ENDOG_OLD:
    p = acorr_ljungbox(v1.resid[col], lags=[8], return_df=True)['lb_pvalue'].values[0]
    print(f'  {col:<32} p={p:.4f}  {"FAIL" if p<0.05 else "pass"}')

print()
# Exog transmission check
print('Exogenous coefficients (LERS + trade channel):')
key_pairs = [
    ('us_ffr',    'hibor_3m',            'us_ffr → hibor (LERS)'),
    ('china_gdp', 'hk_exports_china_yoy','china_gdp → exports (trade)'),
]
for exog_var, eq, label in key_pairs:
    coef = v1.params.loc[exog_var, eq]
    se   = v1.stderr.loc[exog_var, eq]
    pval  = v1.pvalues.loc[exog_var, eq]
    sig   = '***' if pval<0.01 else ('**' if pval<0.05 else ('*' if pval<0.10 else ''))
    print(f'  {label:<35} coef={coef:+.3f}  p={pval:.3f}  {sig}')

print()
# IRF comparison table — key channels at h=1,2
irf_v1 = v1.irf(periods=8)
lo_v1, hi_v1 = irf_v1.errband_mc(orth=True, repl=1000, signif=0.10)
si = {v: i for i, v in enumerate(ENDOG_OLD)}
ri = {v: i for i, v in enumerate(ENDOG_OLD)}
channels = [
    (si['hibor_3m'], ri['hk_property_price_qoq'], 'hibor → property'),
    (si['hk_exports_china_yoy'], ri['gdp_growth'], 'exports → gdp'),
    (si['gdp_growth'], ri['cpi_inflation'],         'gdp → cpi'),
]
print('Bootstrap IRF (90% MC, 1000 repl):')
print(f'{"Channel":<22} {"h":>3}  CI')
print('-'*55)
for imp, resp, label in channels:
    for h in [1, 2, 4]:
        lo = lo_v1[h, resp, imp]; hi = hi_v1[h, resp, imp]
        sig = 'sig' if not (lo < 0 < hi) else 'x0'
        print(f'{label:<22} {h:>3}  ({lo:+.3f}, {hi:+.3f}) {sig}')
    print()

# FEVD headline
fevd_v1 = v1.fevd(periods=8).decomp
print(f'FEVD at h=8: HIBOR→property={fevd_v1[ri["hk_property_price_qoq"]][7][si["hibor_3m"]]*100:.0f}%  |  exports→gdp={fevd_v1[ri["gdp_growth"]][7][si["hk_exports_china_yoy"]]*100:.0f}%')

# Save full IRF figure
fig = irf_v1.plot(orth=True, stderr_type='mc', repl=1000, signif=0.1)
fig.savefig('output/phase3_irf_bootstrap_full.png', dpi=80, bbox_inches='tight')
plt.close()
print('\nSaved: output/phase3_irf_bootstrap_full.png')

VARX(1) | obs=112 | AIC=4.3349 | BIC=5.6456

Ljung-Box (lag=8):
  hk_exports_china_yoy             p=0.0888  pass
  gdp_growth                       p=0.0000  FAIL
  cpi_inflation                    p=0.0012  FAIL
  unemployment                     p=0.1697  pass
  hibor_3m                         p=0.9652  pass
  hk_property_price_qoq            p=0.0789  pass

Exogenous coefficients (LERS + trade channel):
  us_ffr → hibor (LERS)               coef=+0.511  p=0.000  ***
  china_gdp → exports (trade)         coef=+0.806  p=0.014  **



Bootstrap IRF (90% MC, 1000 repl):
Channel                  h  CI
-------------------------------------------------------
hibor → property         1  (-1.683, -0.442) sig
hibor → property         2  (-0.988, -0.089) sig
hibor → property         4  (-0.322, +0.129) x0

exports → gdp            1  (+0.121, +0.827) sig
exports → gdp            2  (-0.165, +0.525) x0
exports → gdp            4  (-0.265, +0.220) x0

gdp → cpi                1  (+0.041, +0.327) sig
gdp → cpi                2  (+0.163, +0.496) sig
gdp → cpi                4  (+0.269, +0.687) sig

FEVD at h=8: HIBOR→property=24%  |  exports→gdp=17%



Saved: output/phase3_irf_bootstrap_full.png


### VARX(4) rejected — OLS overfit

In [ ]:
# VARX(4): why it was rejected — OLS overfit, exports→gdp disappears
# 27 params/eq on 107 obs = 3.9:1 ratio. OLS memorises noise, CIs explode.

df = pd.read_csv(DATA, index_col=0, parse_dates=True)

v1_c = VAR(df[ENDOG_OLD], exog=df[EXOG]).fit(maxlags=1)
v4_c = VAR(df[ENDOG_OLD], exog=df[EXOG]).fit(maxlags=4)

k1 = 1 * len(ENDOG_OLD) + len(EXOG) + 1   # 9 params/eq
k4 = 4 * len(ENDOG_OLD) + len(EXOG) + 1   # 27 params/eq
nobs = v4_c.nobs

print(f'Model comparison:')
print(f'  VARX(1): {k1} params/eq  |  BIC={v1_c.bic:.4f}')
print(f'  VARX(4): {k4} params/eq  |  BIC={v4_c.bic:.4f}  (BIC +{v4_c.bic - v1_c.bic:.2f})')
print(f'  VARX(4) obs={nobs}  params/obs = {k4}/{nobs} = {k4/nobs:.2f}')
print()

irf1 = v1_c.irf(periods=8)
irf4 = v4_c.irf(periods=8)
lo1, hi1 = irf1.errband_mc(orth=True, repl=500, signif=0.10)
lo4, hi4 = irf4.errband_mc(orth=True, repl=500, signif=0.10)

si = {v: i for i, v in enumerate(ENDOG_OLD)}
channels = [
    (si['hibor_3m'],              si['hk_property_price_qoq'], 'hibor → property'),
    (si['hk_exports_china_yoy'],  si['gdp_growth'],            'exports → gdp'),
]

print('Bootstrap IRF (90% MC, 500 repl):')
print(f'{"Channel":<22} {"h":>3}  {"VARX(1)":>22}  {"VARX(4)":>22}')
print('-' * 78)
for imp, resp, label in channels:
    for h in [1, 2, 4]:
        l1, h1 = lo1[h, resp, imp], hi1[h, resp, imp]
        l4, h4 = lo4[h, resp, imp], hi4[h, resp, imp]
        s1 = 'sig' if not (l1 < 0 < h1) else 'x0 '
        s4 = 'sig' if not (l4 < 0 < h4) else 'x0 '
        print(f'{label:<22} {h:>3}  ({l1:+.3f},{h1:+.3f}) {s1}  ({l4:+.3f},{h4:+.3f}) {s4}')
    print()

# CI width comparison — shows variance explosion
BLUE = '#1a6faf'; RED = '#c0392b'
t = np.arange(9)
fig, axes = plt.subplots(1, 2, figsize=(12, 4)); fig.patch.set_facecolor('white')
for ax, (imp, resp, title) in zip(axes, channels):
    w1 = hi1[:, resp, imp] - lo1[:, resp, imp]
    w4 = hi4[:, resp, imp] - lo4[:, resp, imp]
    ax.plot(t, w1, color=BLUE, lw=2, label='VARX(1)')
    ax.plot(t, w4, color=RED, lw=2, ls='--', label='VARX(4)')
    ax.set_title(f'{title} — 90% CI width', fontsize=9, fontweight='bold')
    ax.set_xlabel('Quarters'); ax.legend(fontsize=8); ax.tick_params(labelsize=8)
    ax.axhline(0, color='k', lw=0.5, ls=':')
fig.suptitle('VARX(4) rejected: CI width explodes from OLS overfit (BIC +1.65 over VARX(1))', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig('output/varx4_rejection.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/varx4_rejection.png')


### BVAR(4) comparison


In [4]:
# BVAR(4) vs VARX(1) comparison

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y4 = df[ENDOG_OLD].values.astype(float)
X  = df[EXOG].values.astype(float)
idx4 = {v: i for i, v in enumerate(ENDOG_OLD)}

bvar4 = MinnesotaBayesianVar(
    endogenous=Y4, exogenous=X, lags=4,
    pi1=0.1, pi2=0.5, pi3=1,
    ar_coefficients=AR_COEF_OLD,
    iterations=2000, credibility_level=0.90, verbose=False
)
bvar4.estimate()
irf4, _ = bvar4.impulse_response_function(h=9, credibility_level=0.90)
fevd4   = bvar4.forecast_error_variance_decomposition(h=8, credibility_level=0.90)

channels = [
    (idx4['hibor_3m'], idx4['hk_property_price_qoq'], 'hibor → property'),
    (idx4['hk_exports_china_yoy'], idx4['gdp_growth'], 'exports → gdp'),
    (idx4['gdp_growth'], idx4['cpi_inflation'],         'gdp → cpi'),
]

# Reload VARX(1) IRF for side-by-side
v1_reload = VAR(df[ENDOG_OLD], exog=df[EXOG]).fit(maxlags=1)
irf_v1r   = v1_reload.irf(periods=8)
lo_v1r, hi_v1r = irf_v1r.errband_mc(orth=True, repl=500, signif=0.10)

print('IRF comparison — VARX(1) vs BVAR(4) Litterman (90% bands)')
print(f'{"Channel":<20} {"h":>3}  {"VARX(1)":>20}  {"BVAR(4)":>20}')
print('-'*70)
for imp, resp, label in channels:
    for h in [1, 2, 4]:
        v1_lo = lo_v1r[h, resp, imp]; v1_hi = hi_v1r[h, resp, imp]
        b4_lo = irf4[resp, imp, h, 1]; b4_hi = irf4[resp, imp, h, 2]
        v1_s  = 'sig' if not (v1_lo < 0 < v1_hi) else 'x0'
        b4_s  = 'sig' if not (b4_lo < 0 < b4_hi) else 'x0'
        print(f'{label:<20} {h:>3}  ({v1_lo:+.3f},{v1_hi:+.3f}){v1_s:>4}  ({b4_lo:+.3f},{b4_hi:+.3f}){b4_s:>4}')
    print()

print('FEVD at h=8 (BVAR(4) Litterman):')
print(f'  HIBOR share in property: {fevd4[idx4["hk_property_price_qoq"],idx4["hibor_3m"],7,0]*100:.0f}%')
print(f'  Exports share in GDP:    {fevd4[idx4["gdp_growth"],idx4["hk_exports_china_yoy"],7,0]*100:.0f}%')
print(f'  HIBOR own-share:         {fevd4[idx4["hibor_3m"],idx4["hibor_3m"],7,0]*100:.0f}%')

# Plot comparison
BLUE = '#1a6faf'; RED = '#c0392b'
t = np.arange(9)
fig, axes = plt.subplots(1, 3, figsize=(14, 4)); fig.patch.set_facecolor('white')
for ax, (imp, resp, title) in zip(axes, channels):
    ax.plot(t, irf_v1r.orth_irfs[:, resp, imp], color=BLUE, lw=2, label='VARX(1)')
    ax.fill_between(t, lo_v1r[:, resp, imp], hi_v1r[:, resp, imp], alpha=0.15, color=BLUE)
    ax.plot(t, irf4[resp, imp, :, 0], color=RED, lw=2, ls='--', label='BVAR(4)')
    ax.fill_between(t, irf4[resp, imp, :, 1], irf4[resp, imp, :, 2], alpha=0.15, color=RED)
    ax.axhline(0, color='k', lw=0.7, ls=':')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('Quarters', fontsize=8); ax.legend(fontsize=8); ax.tick_params(labelsize=8)
fig.suptitle('VARX(1) vs BVAR(4) Litterman (pi1=0.1) | exports→gdp recovered by BVAR', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig('output/phase4_bvar_irf_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/phase4_bvar_irf_comparison.png')

Phase 4: IRF comparison — VARX(1) vs BVAR(4) Litterman (90% bands)
Channel                h               VARX(1)               BVAR(4)
----------------------------------------------------------------------
hibor → property       1  (-1.654,-0.442) sig  (-0.635,-0.229) sig
hibor → property       2  (-0.985,-0.121) sig  (-0.303,+0.081)  x0
hibor → property       4  (-0.333,+0.121)  x0  (-0.117,+0.089)  x0

exports → gdp          1  (+0.117,+0.868) sig  (+0.299,+0.514) sig
exports → gdp          2  (-0.172,+0.585)  x0  (+0.119,+0.379) sig
exports → gdp          4  (-0.278,+0.260)  x0  (-0.135,+0.105)  x0

gdp → cpi              1  (+0.035,+0.326) sig  (+0.091,+0.177) sig
gdp → cpi              2  (+0.166,+0.500) sig  (+0.134,+0.254) sig
gdp → cpi              4  (+0.280,+0.697) sig  (+0.178,+0.320) sig

FEVD at h=8 (BVAR(4) Litterman):
  HIBOR share in property: 7%
  Exports share in GDP:    19%
  HIBOR own-share:         85%


Saved: output/phase4_bvar_irf_comparison.png


### Hyperparameters and fit


In [5]:
# Lag-length ML grid — ENDOG_OLD ordering

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y5 = df[ENDOG_OLD].values.astype(float)
X  = df[EXOG].values.astype(float)

print('Lag-length grid — ML-optimized prior (iterations=500 for speed):')
print(f'{"p":>4}  {"opt_pi1":>8}  {"opt_pi2":>8}  {"ML":>12}  {"LB fail":>8}')
print('-' * 52)
opt_results = {}
for p in range(1, 6):
    bv = MinnesotaBayesianVar(
        endogenous=Y5, exogenous=X, lags=p,
        hyperparameter_optimization=True,
        iterations=500, credibility_level=0.90, verbose=False
    )
    bv.estimate()
    ml = bv.marginal_likelihood()
    bv.insample_fit()
    resid = bv.residual_estimates[:, :, 0]
    lb_fails = sum(
        acorr_ljungbox(resid[:, i], lags=[8], return_df=True)['lb_pvalue'].values[0] < 0.05
        for i in range(len(ENDOG_OLD))
    )
    opt_results[p] = {'ml': ml, 'pi1': bv.pi1, 'pi2': bv.pi2, 'lb': lb_fails}
    print(f'{p:>4}  {bv.pi1:>8.4f}  {bv.pi2:>8.4f}  {ml:>12.4f}  {lb_fails:>5}/6')

best_p = max(opt_results, key=lambda p: opt_results[p]['ml'])
print(f'\nML selects p={best_p} | pi1={opt_results[best_p]["pi1"]:.4f} | pi2={opt_results[best_p]["pi2"]:.4f}')
print(f'BVAR(4): ML={opt_results[4]["ml"]:.2f} | pi1={opt_results[4]["pi1"]:.4f} | LB={opt_results[4]["lb"]}/6')
print()
print('Note: p=3 optimizer may degenerate to pi1=0.01 (extremely tight prior — data ignored).')
print('      BVAR(4) is the best joint ML+diagnostic model (pi1~0.085, 2/6 LB fail).')
print()
print(f'FINAL SPEC confirmed: p=4, pi1={PI1}, pi2={PI2}, pi3={PI3}')
print(f'AR_COEF_OLD (ML-optimal delta): {AR_COEF_OLD}  [exp, gdp, cpi, unemp, hibor, prop]')

Lag-length grid — ML-optimized prior (iterations=500 for speed):
   p   opt_pi1   opt_pi2            ML   LB fail
----------------------------------------------------


   1    0.0481    1.0000     -539.7396      3/6


   2    0.0449    1.0000     -551.3241      3/6


   3    0.0471    1.0000     -534.4644      3/6


   4    0.0855    1.0000     -534.2911      2/6


   5    0.1013    1.0000     -532.6522      2/6

ML selects p=5 | pi1=0.1013 | pi2=1.0000
BVAR(4): ML=-534.29 | pi1=0.0855 | LB=2/6

Note: p=3 optimizer may degenerate to pi1=0.01 (extremely tight prior — data ignored).
      BVAR(4) is the best joint ML+diagnostic model (pi1~0.085, 2/6 LB fail).

FINAL SPEC confirmed: p=4, pi1=0.085, pi2=1.0, pi3=1.0
AR_COEF_OLD (ML-optimal delta): [0.627, 0.545, 0.735, 0.991, 0.442, 0.418]  [exp, gdp, cpi, unemp, hibor, prop]


### Prior sensitivity: pi1 × pi3 grid

In [ ]:
# Prior sensitivity: pi1 × pi3 marginal likelihood grid
# pi1 = own-lag shrinkage strength; pi3 = lag decay rate
# Shows why pi1≈0.085 and pi3=1.0 were chosen.

import matplotlib.colors as mcolors

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y = df[ENDOG_OLD].values.astype(float)
X = df[EXOG].values.astype(float)

pi1_grid = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
pi3_grid = [0.3, 0.5, 1.0, 2.0]

ml_grid = np.full((len(pi3_grid), len(pi1_grid)), np.nan)
for i, pi3_val in enumerate(pi3_grid):
    for j, pi1_val in enumerate(pi1_grid):
        bv = MinnesotaBayesianVar(
            endogenous=Y, exogenous=X, lags=LAGS,
            pi1=pi1_val, pi2=PI2, pi3=pi3_val,
            ar_coefficients=AR_COEF_OLD,
            iterations=500, credibility_level=0.90, verbose=False
        )
        bv.estimate()
        ml_grid[i, j] = bv.marginal_likelihood()

print('Marginal likelihood: rows=pi3, cols=pi1')
print(f'{"pi3\\pi1":<8}' + ''.join(f'{v:>9.2f}' for v in pi1_grid))
print('-' * (8 + 9 * len(pi1_grid)))
for i, pi3_val in enumerate(pi3_grid):
    print(f'{pi3_val:<8}' + ''.join(f'{ml_grid[i,j]:>9.2f}' for j in range(len(pi1_grid))))

best_i, best_j = np.unravel_index(np.nanargmax(ml_grid), ml_grid.shape)
print(f'\nManual grid best: pi1={pi1_grid[best_j]}, pi3={pi3_grid[best_i]}, ML={ml_grid[best_i, best_j]:.2f}')
print(f'ML-optimizer (with delta freedom): pi1=0.085, pi3=1.0, ML=-528.98 — optimizer dominates')
print('pi2=1.0 universal: no cross-variable shrinkage at any pi1/pi3 combination')

fig, ax = plt.subplots(figsize=(8, 4)); fig.patch.set_facecolor('white')
cmap = mcolors.LinearSegmentedColormap.from_list('ml', ['#c0392b', '#f5f5f5', '#1a6faf'])
im = ax.imshow(ml_grid, cmap=cmap, aspect='auto')
ax.set_xticks(range(len(pi1_grid)));  ax.set_xticklabels([str(v) for v in pi1_grid], fontsize=9)
ax.set_yticks(range(len(pi3_grid)));  ax.set_yticklabels([str(v) for v in pi3_grid], fontsize=9)
ax.set_xlabel('pi1  (own-lag shrinkage)', fontsize=10)
ax.set_ylabel('pi3  (lag decay)', fontsize=10)
for i in range(len(pi3_grid)):
    for j in range(len(pi1_grid)):
        ax.text(j, i, f'{ml_grid[i,j]:.1f}', ha='center', va='center', fontsize=8)
ax.set_title('Prior sensitivity: pi1 × pi3 marginal likelihood  (p=4, pi2=1.0)', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, label='Marginal likelihood', shrink=0.85)
plt.tight_layout()
plt.savefig('output/prior_sensitivity_pi1_pi3.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/prior_sensitivity_pi1_pi3.png')


In [6]:
# Posterior distribution verification
# Minnesota BVAR draws are i.i.d. analytical posterior draws, not MCMC.

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y5 = df[ENDOG_OLD].values.astype(float)
X = df[EXOG].values.astype(float)
idx_old = {v: i for i, v in enumerate(ENDOG_OLD)}

bv5b = MinnesotaBayesianVar(
    endogenous=Y5, exogenous=X, lags=LAGS,
    pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_OLD,
    iterations=5000, credibility_level=0.90, verbose=False
)
bv5b.estimate()
draws = bv5b.mcmc_beta

regressors = ['const'] + EXOG
for lag in range(1, LAGS + 1):
    regressors.extend([f'L{lag}.{var}' for var in ENDOG_OLD])
reg_idx = {name: i for i, name in enumerate(regressors)}

monitor = [
    ('us_ffr', 'hibor_3m', 'us_ffr → hibor', '+'),
    ('china_gdp', 'hk_exports_china_yoy', 'china_gdp → exports', '+'),
    ('L1.hk_exports_china_yoy', 'gdp_growth', 'L1.exports → gdp', '+'),
    ('L1.gdp_growth', 'cpi_inflation', 'L1.gdp → cpi', '+'),
    ('L1.hibor_3m', 'hk_property_price_qoq', 'L1.hibor → property', '-'),
    ('const', 'gdp_growth', 'const → gdp', '?'),
]

print('Posterior check (5000 analytical draws):')
print(f'{"Parameter":<28} {"Mean":>8} {"Std":>8} {"Sign":>8}')
print('-' * 58)
for reg_name, eq_name, label, expected in monitor:
    chain = draws[reg_idx[reg_name], idx_old[eq_name], :]
    mean = chain.mean()
    std = chain.std()
    if expected == '+':
        sign = 'YES' if mean > 0 else 'FAIL'
    elif expected == '-':
        sign = 'YES' if mean < 0 else 'FAIL'
    else:
        sign = 'N/A'
    print(f'{label:<28} {mean:>8.4f} {std:>8.4f} {sign:>8}')

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.patch.set_facecolor('white')
for ax, (reg_name, eq_name, label, expected) in zip(axes.flat, monitor):
    chain = draws[reg_idx[reg_name], idx_old[eq_name], :]
    ax.hist(chain, bins=60, color='#1a6faf', alpha=0.7, density=True)
    ax.axvline(chain.mean(), color='red', lw=1.5, ls='--', label=f'mean={chain.mean():.3f}')
    ax.axvline(0, color='black', lw=0.8, ls=':')
    ax.set_title(f'{label} (expected: {expected})', fontsize=8, fontweight='bold')
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)
fig.suptitle('Posterior verification — analytical draws, not MCMC', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig('output/phase5_posterior_verification.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/phase5_posterior_verification.png')


Posterior check (5000 analytical draws):
Parameter                        Mean      Std     Sign
----------------------------------------------------------
us_ffr → hibor                 0.5265   0.0595      YES
china_gdp → exports            0.8395   0.2412      YES
L1.exports → gdp              -0.0004   0.0136     FAIL
L1.gdp → cpi                   0.0309   0.0234      YES
L1.hibor → property           -0.5403   0.4035      YES
const → gdp                   -2.2424   0.9375      N/A


Saved: output/phase5_posterior_verification.png


In [7]:
# In-sample fit diagnostics

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y5 = df[ENDOG_OLD].values.astype(float)
X  = df[EXOG].values.astype(float)

bv5c = MinnesotaBayesianVar(
    endogenous=Y5, exogenous=X, lags=LAGS,
    pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_OLD,
    iterations=2000, credibility_level=0.90, verbose=False
)
bv5c.estimate()
bv5c.insample_fit()
resid5 = bv5c.residual_estimates[:, :, 0]
T_eff  = resid5.shape[0]
Y_tr   = Y5[-T_eff:]
fitted = Y_tr - resid5

print(f'In-sample fit — BVAR(4) pi1={PI1} pi2={PI2} AR_COEF_OLD=ML-optimal:')
print(f'{"Variable":<32} {"R2":>6} {"RMSE":>8} {"LB p":>8} {"Status":>8}')
print('-' * 65)
for i, col in enumerate(ENDOG_OLD):
    ss_res = np.sum(resid5[:, i]**2)
    ss_tot = np.sum((Y_tr[:, i] - Y_tr[:, i].mean())**2)
    r2   = 1 - ss_res / ss_tot
    rmse = np.sqrt(ss_res / T_eff)
    lb_p = acorr_ljungbox(resid5[:, i], lags=[8], return_df=True)['lb_pvalue'].values[0]
    print(f'{col:<32} {r2:>6.3f} {rmse:>8.3f} {lb_p:>8.4f} {"FAIL" if lb_p<0.05 else "pass":>8}')

# Plot fitted vs actual for key variables
focus = ['gdp_growth', 'cpi_inflation', 'hibor_3m']
fig, axes = plt.subplots(3, 3, figsize=(14, 9)); fig.patch.set_facecolor('white')
for row, col in enumerate(focus):
    i = ENDOG_OLD.index(col)
    ax0, ax1, ax2 = axes[row]
    ax0.plot(Y_tr[:, i], 'k-', lw=1.2, label='Actual')
    ax0.plot(fitted[:, i], '#c0392b', lw=1.2, ls='--', label='Fitted')
    ax0.set_title(f'{col} — fitted vs actual', fontsize=8, fontweight='bold'); ax0.legend(fontsize=7)
    ax1.plot(resid5[:, i], '#1a6faf', lw=0.8); ax1.axhline(0, color='k', lw=0.5, ls=':')
    ax1.set_title('Residuals', fontsize=8)
    plot_acf(resid5[:, i], lags=16, ax=ax2, alpha=0.05); ax2.set_title('ACF', fontsize=8)
    for ax in [ax0, ax1, ax2]: ax.tick_params(labelsize=7)
plt.suptitle('In-sample fit — BVAR(4) pi1=0.085 pi2=1.0 (ML-optimal AR_COEF_OLD)\n2/6 LB fail = structural breaks, not lag misspecification', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig('output/phase5_insample_fit.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/phase5_insample_fit.png')

In-sample fit — BVAR(4) pi1=0.085 pi2=1.0 AR_COEF_OLD=ML-optimal:
Variable                             R2     RMSE     LB p   Status
-----------------------------------------------------------------
hk_exports_china_yoy              0.656    7.314   0.2142     pass
gdp_growth                        0.792    1.795   0.0000     FAIL
cpi_inflation                     0.915    0.744   0.0014     FAIL
unemployment                      0.949    0.331   0.5906     pass
hibor_3m                          0.953    0.416   0.2772     pass
hk_property_price_qoq             0.412    3.401   0.4278     pass


Saved: output/phase5_insample_fit.png


In [8]:
# Conditional OOS RMSE
# Realized future exogenous values are used for both models.

def _fixed_forecast_regressors(Z_p, Y, h, p, T, exogenous, constant, trend, quadratic_trend):
    temp = vu.generate_intercept_and_trends(constant, trend, quadratic_trend, h, T)
    exog_empty = (exogenous is None or
                  (isinstance(exogenous, list) and len(exogenous) == 0) or
                  (isinstance(exogenous, np.ndarray) and exogenous.size == 0))
    if exog_empty:
        Z_p = []
    elif isinstance(Z_p, list) and len(Z_p) == 0:
        Z_p = np.tile(exogenous[-1], [h, 1])
    if len(Z_p) != 0:
        Z_p = np.hstack([temp, Z_p])
    elif any([constant, trend, quadratic_trend]):
        Z_p = temp
    else:
        Z_p = []
    Y = Y[-p:, :]
    return Z_p, Y
vu.make_forecast_regressors = _fixed_forecast_regressors

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y_all = df[ENDOG_OLD].values.astype(float)
X_all = df[EXOG].values.astype(float)
T = len(Y_all); T_start = 85; horizons = [1, 2, 4]

v1_preds = {h: [] for h in horizons}
bv_preds = {h: [] for h in horizons}
actuals  = {h: [] for h in horizons}

t0 = time.time()
for t in range(T_start, T):
    df_tr = df.iloc[:t]; Y_tr = Y_all[:t]; X_tr = X_all[:t]
    m1 = VAR(df_tr[ENDOG_OLD], exog=df_tr[EXOG]).fit(maxlags=1)
    bv = MinnesotaBayesianVar(
        endogenous=Y_tr, exogenous=X_tr, lags=LAGS,
        pi1=PI1, pi2=PI2, pi3=PI3,
        ar_coefficients=AR_COEF_OLD,
        iterations=300, credibility_level=0.90, verbose=False
    )
    bv.estimate()
    for h in horizons:
        if t + h > T: continue
        X_fut = X_all[t:t+h]
        fc1   = m1.forecast(df_tr[ENDOG_OLD].values[-1:], steps=h, exog_future=X_fut)
        fc_bv = bv.forecast(h=h, credibility_level=0.90, Z_p=X_fut)
        v1_preds[h].append(fc1[h-1])
        bv_preds[h].append(fc_bv[h-1, :, 0])
        actuals[h].append(Y_all[t+h-1])
print(f'OOS done: {time.time()-t0:.1f}s | {T-T_start} windows')

print('\nConditional OOS RMSE ratio (BVAR/VARX1 — <1 = BVAR wins):')
print(f'{"Variable":<28} {"h=1":>8}  {"h=2":>8}  {"h=4":>8}')
print('-' * 58)
wins = 0
for i, col in enumerate(ENDOG_OLD):
    ratios = []
    for h in horizons:
        act = np.array(actuals[h])[:,i]
        rv1 = np.sqrt(np.nanmean((np.array(v1_preds[h])[:,i]-act)**2))
        rbv = np.sqrt(np.nanmean((np.array(bv_preds[h])[:,i]-act)**2))
        r   = rbv/rv1
        ratios.append(r)
        if r < 1: wins += 1
    stars = ['**' if r < 1 else '  ' for r in ratios]
    print(f'{col:<28} {ratios[0]:>6.3f}{stars[0]}  {ratios[1]:>6.3f}{stars[1]}  {ratios[2]:>6.3f}{stars[2]}')
print(f'\nBVAR wins {wins}/18 cells (** = BVAR wins)')

OOS done: 0.6s | 28 windows

Conditional OOS RMSE ratio (BVAR/VARX1 — <1 = BVAR wins):
Variable                          h=1       h=2       h=4
----------------------------------------------------------
hk_exports_china_yoy          0.898**   0.913**   0.952**
gdp_growth                    0.891**   0.901**   0.897**
cpi_inflation                 0.986**   0.942**   0.822**
unemployment                  0.968**   0.914**   0.839**
hibor_3m                      0.820**   0.798**   0.830**
hk_property_price_qoq         0.888**   0.913**   0.906**

BVAR wins 18/18 cells (** = BVAR wins)


## Data checks


In [9]:
# ADF + KPSS stationarity battery

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
prop_idx = df['hk_property_price_idx']

test_series = {
    'hk_exports_china_yoy':  df['hk_exports_china_yoy'],
    'gdp_growth':            df['gdp_growth'],
    'cpi_inflation':         df['cpi_inflation'],
    'unemployment':          df['unemployment'],
    'hibor_3m':              df['hibor_3m'],
    'hk_property_price_qoq': df['hk_property_price_qoq'],
    'hk_property_price_idx': prop_idx,
}

print('ADF + KPSS stationarity audit:')
print(f'{"Series":<28} {"ADF_p":>8}  {"KPSS_p":>8}  Verdict')
print('-' * 65)
i1_candidates = []
for name, s in test_series.items():
    s_clean = s.dropna()
    adf_p = adfuller(s_clean, autolag='BIC')[1]
    try:
        kpss_stat, kpss_p, _, _ = kpss(s_clean, regression='c', nlags='auto')
        kpss_p_display = kpss_p
    except Exception:
        kpss_p_display = np.nan
    # I(1) if both tests agree: ADF fails to reject AND KPSS rejects
    if adf_p > 0.05 and (np.isnan(kpss_p_display) or kpss_p_display < 0.05):
        verdict = 'I(1)'
        i1_candidates.append(name)
    elif adf_p < 0.05 and (np.isnan(kpss_p_display) or kpss_p_display > 0.05):
        verdict = 'I(0)'
    else:
        verdict = 'ambiguous'
    print(f'{name:<28} {adf_p:>8.4f}  {kpss_p_display if not np.isnan(kpss_p_display) else "nan":>8}  {verdict}')

print(f'\nI(1) candidates: {i1_candidates}')
print('Endogenous I(1) block for Johansen: [unemployment, cpi_inflation, hk_property_price_idx]')


ADF + KPSS stationarity audit:
Series                          ADF_p    KPSS_p  Verdict
-----------------------------------------------------------------
hk_exports_china_yoy           0.0000       0.1  I(0)
gdp_growth                     0.0161       0.1  I(0)
cpi_inflation                  0.1526  0.020207717362126263  I(1)
unemployment                   0.0821      0.01  I(1)
hibor_3m                       0.0368  0.04945221856936502  ambiguous
hk_property_price_qoq          0.0000  0.08304920321603894  I(0)
hk_property_price_idx          0.8222      0.01  I(1)

I(1) candidates: ['cpi_inflation', 'unemployment', 'hk_property_price_idx']
Endogenous I(1) block for Johansen: [unemployment, cpi_inflation, hk_property_price_idx]


### Delta-u robustness


In [10]:
# Δu robustness

df_du = df.copy()
df_du['unemployment'] = df['unemployment'].diff()
df_du = df_du.dropna()

AR_COEF_DU = AR_COEF_OLD.copy()
AR_COEF_DU[ENDOG_OLD.index('unemployment')] = df_du['unemployment'].autocorr(lag=1)
print(f"Δu AR coefficient: {AR_COEF_DU[ENDOG_OLD.index('unemployment')]:.3f}")

bvar_base = MinnesotaBayesianVar(
    endogenous=df[ENDOG_OLD].values, exogenous=df[EXOG].values,
    lags=LAGS, pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_OLD,
    iterations=3000, credibility_level=0.90, verbose=False
)
bvar_base.estimate()
irf_base, _ = bvar_base.impulse_response_function(h=9, credibility_level=0.90)

bvar_du = MinnesotaBayesianVar(
    endogenous=df_du[ENDOG_OLD].values, exogenous=df_du[EXOG].values,
    lags=LAGS, pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_DU,
    iterations=3000, credibility_level=0.90, verbose=False
)
bvar_du.estimate()
irf_du, _ = bvar_du.impulse_response_function(h=9, credibility_level=0.90)

idx4 = {v: i for i, v in enumerate(ENDOG_OLD)}
checks = [('HIBOR → Property', idx4['hibor_3m'], idx4['hk_property_price_qoq']),
          ('Exports → GDP',    idx4['hk_exports_china_yoy'], idx4['gdp_growth'])]

print('IRF at h=2 — baseline vs Δu robustness:')
print(f'{"Channel":<22} {"Base CI":>22}  {"Δu CI":>22}  Verdict')
print('-' * 75)
for label, imp, resp in checks:
    b_lo = irf_base[resp, imp, 2, 1]; b_hi = irf_base[resp, imp, 2, 2]
    d_lo = irf_du[resp, imp, 2, 1];   d_hi = irf_du[resp, imp, 2, 2]
    b_s  = 'sig' if not (b_lo < 0 < b_hi) else 'x0'
    d_s  = 'sig' if not (d_lo < 0 < d_hi) else 'x0'
    verdict = 'PASS' if b_s == d_s else 'CHANGED'
    print(f'{label:<22} ({b_lo:+.3f},{b_hi:+.3f}) {b_s}  ({d_lo:+.3f},{d_hi:+.3f}) {d_s}  {verdict}')

print('\nConclusion: Keep unemployment in levels. One limitation sentence required.')


--- Phase 7.2c: Δu robustness ---
Δu AR coefficient: 0.471


IRF at h=2 — baseline vs Δu robustness:
Channel                               Base CI                   Δu CI  Verdict
---------------------------------------------------------------------------
HIBOR → Property       (-0.442,+0.103) x0  (-0.460,+0.072) x0  PASS
Exports → GDP          (+0.085,+0.410) sig  (+0.052,+0.374) sig  PASS

Conclusion: Keep unemployment in levels. One limitation sentence required.


### Johansen check


In [11]:
# Johansen cointegration — endogenous I(1) block only

df    = pd.read_csv(DATA, index_col=0, parse_dates=True)
prop_idx = df['hk_property_price_idx']

# I(1) endogenous block — exogenous excluded deliberately
i1_block = pd.DataFrame({
    'unemployment':         df['unemployment'],
    'cpi_inflation':        df['cpi_inflation'],
    'hk_property_price_idx': prop_idx,
}).dropna()

print(f'I(1) endogenous block: n={len(i1_block)} obs, variables: {list(i1_block.columns)}')
print('Note: us_ffr and china_gdp EXCLUDED — LERS peg is not cointegration')
print()

res = coint_johansen(i1_block, det_order=0, k_ar_diff=1)
print('Johansen results:')
print(f'  Trace statistics:      {[round(float(x),2) for x in res.lr1]}')
print(f'  Trace CV 95%:          {[round(float(x),2) for x in res.cvt[:, 1]]}')
print(f'  Max-eigenvalue stats:  {[round(float(x),2) for x in res.lr2]}')
print(f'  Max-eig CV 95%:        {[round(float(x),2) for x in res.cvm[:, 1]]}')
print()

# Determine rank
rank = 0
for i, (stat, cv) in enumerate(zip(res.lr1, res.cvt[:, 1])):
    if stat > cv:
        rank = i + 1
print(f'Cointegrating rank at 95%: {rank}')
if rank == 0:
    print('VECM not warranted. BVAR-in-levels is the appropriate framework.')
    print('This is a research finding: I(1) variables in this system do not form stable long-run relationships.')
else:
    print(f'Rank={rank} — consider BVECM. Check if driven by exogenous contamination.')

I(1) endogenous block: n=113 obs, variables: ['unemployment', 'cpi_inflation', 'hk_property_price_idx']
Note: us_ffr and china_gdp EXCLUDED — LERS peg is not cointegration

Johansen results:
  Trace statistics:      [np.float64(25.66), np.float64(9.43), np.float64(0.55)]
  Trace CV 95%:          [np.float64(29.8), np.float64(15.49), np.float64(3.84)]
  Max-eigenvalue stats:  [np.float64(16.22), np.float64(8.88), np.float64(0.55)]
  Max-eig CV 95%:        [np.float64(21.13), np.float64(14.26), np.float64(3.84)]

Cointegrating rank at 95%: 0
VECM not warranted. BVAR-in-levels is the appropriate framework.
This is a research finding: I(1) variables in this system do not form stable long-run relationships.


## Robustness checks


In [ ]:
# Structural breaks: VARX(1) vs BVAR(4)
# VARX(1) residuals vs canonical BVAR(4) residuals

import ruptures as rpt

# VARX(1) residuals
v1_break = VAR(df[ENDOG_OLD], exog=df[EXOG]).fit(maxlags=1)
resid_v1 = pd.DataFrame(v1_break.resid, columns=ENDOG_OLD)
dates_v1 = df.index[1:]

# BVAR(4) canonical — uses ENDOG/AR_COEF from config cell
bv = MinnesotaBayesianVar(
    endogenous=df[ENDOG].values.astype(float),
    exogenous=df[EXOG].values.astype(float),
    lags=LAGS, pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF,
    iterations=2000, credibility_level=0.90, verbose=False
)
bv.estimate()
bv.insample_fit()
resid_bv = pd.DataFrame(bv.residual_estimates[:, :, 0], index=df.index[LAGS:], columns=ENDOG)

# ── Helpers ──────────────────────────────────────────────────────────────────
def chow_test(series, break_date, dates):
    pre  = series[dates < pd.Timestamp(break_date)]
    post = series[dates >= pd.Timestamp(break_date)]
    if min(len(pre), len(post)) < 5: return np.nan, np.nan
    _, t_p   = stats.ttest_ind(pre, post, equal_var=False)
    _, l_p   = stats.levene(pre, post)
    return t_p, l_p

def bp_breaks(series):
    s = series.values.reshape(-1, 1)
    pen = s.std()**2 * np.log(len(s)) * 3
    algo = rpt.Pelt(model='l2', min_size=8, jump=1).fit(s)
    return len([b for b in algo.predict(pen=pen) if b < len(s)])

breakpoints  = [('GFC 2008Q3', '2008-07-01'), ('COVID 2020Q1', '2020-01-01')]
focus_vars   = ['gdp_growth', 'cpi_inflation', 'hibor_3m', 'hk_property_price_qoq']

# ── Chow table ───────────────────────────────────────────────────────────────
print('Chow Tests (mean break p-value):')
print(f'{"Variable":<28} {"Break":<14}  {"VARX(1)":>10}  {"BVAR(4)":>10}  Verdict')
print('-' * 80)
for col in focus_vars:
    for bp_label, bp_date in breakpoints:
        p_v1, _ = chow_test(resid_v1[col].values, bp_date, dates_v1) if col in ENDOG_OLD else (np.nan, np.nan)
        p_bv, _ = chow_test(resid_bv[col].values, bp_date, resid_bv.index)
        verdict = '<-- ABSORBED' if (not np.isnan(p_v1) and p_v1 < 0.05 and p_bv >= 0.05) else \
                  'both break'   if (not np.isnan(p_v1) and p_v1 < 0.05 and p_bv < 0.05)  else ''
        print(f'{col:<28} {bp_label:<14}  {p_v1:>10.4f}  {p_bv:>10.4f}  {verdict}')
    print()

# ── Bai-Perron table ─────────────────────────────────────────────────────────
print('Bai-Perron — N breaks found:')
print(f'{"Variable":<28}  {"VARX(1)":>8}  {"BVAR(4)":>8}')
print('-' * 50)
for col in focus_vars:
    n_v1 = bp_breaks(resid_v1[col]) if col in ENDOG_OLD else float('nan')
    n_bv = bp_breaks(resid_bv[col])
    print(f'{col:<28}  {str(int(n_v1)) if not np.isnan(n_v1) else "n/a":>8}  {n_bv:>8}')


### Exogenous lag check


In [13]:
# Exogenous lag check: BVAR(4), q=0 vs q=1
# q=1 adds us_ffr_{t-1}; china_gdp lag is excluded.

# Match samples for q=0 and q=1.
df_q1 = df.copy()
df_q1['us_ffr_lag1'] = df_q1['us_ffr'].shift(1)
df_q1 = df_q1.dropna()
print(f'Dataset for exogenous-lag check: {len(df_q1)} obs')

idx = {v: i for i, v in enumerate(ENDOG_CANONICAL)}
EXOG_Q1 = ['us_ffr', 'china_gdp', 'us_ffr_lag1']

bvar_q0 = MinnesotaBayesianVar(
    endogenous=df_q1[ENDOG_CANONICAL].values.astype(float),
    exogenous=df_q1[EXOG].values.astype(float),
    lags=LAGS, pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_CANONICAL,
    iterations=3000, credibility_level=0.90, verbose=False
)
bvar_q0.estimate()
irf_q0, _ = bvar_q0.impulse_response_function(h=9, credibility_level=0.90)

bvar_q1 = MinnesotaBayesianVar(
    endogenous=df_q1[ENDOG_CANONICAL].values.astype(float),
    exogenous=df_q1[EXOG_Q1].values.astype(float),
    lags=LAGS, pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_CANONICAL,
    iterations=3000, credibility_level=0.90, verbose=False
)
bvar_q1.estimate()
irf_q1, _ = bvar_q1.impulse_response_function(h=9, credibility_level=0.90)

bvar_q0.insample_fit()
bvar_q1.insample_fit()
resid_q0 = bvar_q0.residual_estimates[:, :, 0]
resid_q1 = bvar_q1.residual_estimates[:, :, 0]

print('\nLjung-Box at lag=8:')
print(f'{"Variable":<22}  {"q=0 p":>8}  {"q=1 p":>8}  Verdict')
print('-' * 65)
for var in ['gdp_growth', 'cpi_inflation']:
    vi = idx[var]
    p0 = acorr_ljungbox(resid_q0[:, vi], lags=[8], return_df=True)['lb_pvalue'].iloc[0]
    p1 = acorr_ljungbox(resid_q1[:, vi], lags=[8], return_df=True)['lb_pvalue'].iloc[0]
    verdict = 'cleared' if p0 < 0.05 and p1 >= 0.05 else 'still fails'
    print(f'{var:<22}  {p0:>8.4f}  {p1:>8.4f}  {verdict}')

channels = [
    (idx['hibor_3m'], idx['hk_property_price_qoq'], 'hibor→property'),
    (idx['hk_exports_china_yoy'], idx['gdp_growth'], 'exports→gdp'),
    (idx['hibor_3m'], idx['gdp_growth'], 'hibor→gdp'),
    (idx['hk_property_price_qoq'], idx['gdp_growth'], 'property→gdp'),
]

print('\nIRF comparison (90% bands):')
for imp_i, resp_i, label in channels:
    for h in [1, 2, 4]:
        lo0, hi0 = irf_q0[resp_i, imp_i, h, 1], irf_q0[resp_i, imp_i, h, 2]
        lo1, hi1 = irf_q1[resp_i, imp_i, h, 1], irf_q1[resp_i, imp_i, h, 2]
        s0 = 'sig' if not (lo0 < 0 < hi0) else 'x0 '
        s1 = 'sig' if not (lo1 < 0 < hi1) else 'x0 '
        tag = 'changed' if s0.strip() != s1.strip() else ''
        print(f'  {label:<16} h={h}: q0=({lo0:+.3f},{hi0:+.3f}) {s0} | q1=({lo1:+.3f},{hi1:+.3f}) {s1} {tag}')
    print()


Dataset for exogenous-lag check: 112 obs



Ljung-Box at lag=8:
Variable                   q=0 p     q=1 p  Verdict
-----------------------------------------------------------------
gdp_growth                0.0000    0.0001  still fails
cpi_inflation             0.0024    0.0018  still fails

IRF comparison (90% bands):
  hibor→property   h=1: q0=(-0.976,-0.322) sig | q1=(-0.974,-0.309) sig 
  hibor→property   h=2: q0=(-0.531,+0.069) x0  | q1=(-0.574,+0.107) x0  
  hibor→property   h=4: q0=(-0.230,+0.070) x0  | q1=(-0.304,+0.107) x0  

  exports→gdp      h=1: q0=(+0.259,+0.534) sig | q1=(+0.295,+0.569) sig 
  exports→gdp      h=2: q0=(+0.056,+0.383) sig | q1=(+0.097,+0.416) sig 
  exports→gdp      h=4: q0=(-0.195,+0.118) x0  | q1=(-0.169,+0.142) x0  

  hibor→gdp        h=1: q0=(-0.411,-0.060) sig | q1=(-0.356,+0.010) x0  changed
  hibor→gdp        h=2: q0=(-0.495,-0.131) sig | q1=(-0.423,-0.018) sig 
  hibor→gdp        h=4: q0=(-0.296,-0.061) sig | q1=(-0.267,+0.059) x0  changed

  property→gdp     h=1: q0=(+0.331,+0.634) sig

## README figures


In [14]:
# README Fig 1: HIBOR vs US FFR

df = pd.read_csv(DATA, index_col=0, parse_dates=True)

BLUE = '#1a6faf'; RED = '#c0392b'

fig, ax = plt.subplots(figsize=(11, 4)); fig.patch.set_facecolor('white')
ax.plot(df.index, df['hibor_3m'], color=BLUE, lw=1.8, label='HIBOR 3M (HK)')
ax.plot(df.index, df['us_ffr'],   color=RED,  lw=1.8, ls='--', label='US Fed Funds Rate')
ax.axvspan(pd.Timestamp('2008-07-01'), pd.Timestamp('2009-06-30'), alpha=0.08, color='gray')
ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-06-30'), alpha=0.08, color='gray')
ax.axvspan(pd.Timestamp('2009-01-01'), pd.Timestamp('2015-12-31'), alpha=0.06, color='orange')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2022-03-31'), alpha=0.06, color='orange')
ax.set_ylabel('% p.a.', fontsize=10)
ax.set_title(f'HIBOR tracks US Fed Funds Rate — Currency Board (LERS)\n{SAMPLE_LABEL}', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.tick_params(labelsize=9)
ax.set_xlim(df.index[0], df.index[-1])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
corr = df['hibor_3m'].corr(df['us_ffr'])
ax.text(0.02, 0.92, f'corr = {corr:.2f}', transform=ax.transAxes,
        fontsize=10, fontweight='bold', color=BLUE,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=BLUE, alpha=0.8))
plt.tight_layout()
plt.savefig('output/readme_hibor_ffr.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved: output/readme_hibor_ffr.png  (corr={corr:.3f})')


Saved: output/readme_hibor_ffr.png  (corr=0.921)


In [15]:
# README Fig 2: Raw data panel — 6 endogenous variables

df = pd.read_csv(DATA, index_col=0, parse_dates=True)

LABELS = ['HIBOR 3M (%)', 'Exports to China YoY (%)', 'Property QoQ (%)',
          'GDP Growth YoY (%)', 'CPI Inflation YoY (%)', 'Unemployment (%)']
COLORS = ['#1a6faf', '#27ae60', '#8e44ad', '#c0392b', '#e67e22', '#2c3e50']

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
fig.patch.set_facecolor('white')

for ax, col, label, color in zip(axes.flat, ENDOG, LABELS, COLORS):
    ax.plot(df.index, df[col], color=color, lw=1.4)
    ax.axhline(0, color='k', lw=0.5, ls=':')
    ax.axvspan(pd.Timestamp('2008-07-01'), pd.Timestamp('2009-06-30'), alpha=0.10, color='gray')
    ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-03-31'), alpha=0.10, color='gray')
    ax.set_title(label, fontsize=9, fontweight='bold', color=color)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

axes[0,0].text(pd.Timestamp('2008-10-01'), axes[0,0].get_ylim()[1]*0.85, 'GFC', fontsize=7, color='gray', ha='center')
axes[0,0].text(pd.Timestamp('2020-04-01'), axes[0,0].get_ylim()[1]*0.85, 'COVID', fontsize=7, color='gray', ha='center')
fig.suptitle(f'HK Macro Variables | {SAMPLE_LABEL}', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('output/readme_raw_data.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/readme_raw_data.png')


Saved: output/readme_raw_data.png


In [16]:
# README Fig 3: OOS RMSE heatmap
# Re-runs OOS loop.

import matplotlib.colors as mcolors

VAR_LABELS = ['Exports YoY', 'GDP Growth', 'CPI Inflation', 'Unemployment', 'HIBOR 3M', 'Property QoQ']
H_LABELS   = ['h = 1', 'h = 2', 'h = 4']

df    = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y_all = df[ENDOG_OLD].values.astype(float)
X_all = df[EXOG].values.astype(float)
T     = len(Y_all); T_start = 85; horizons = [1, 2, 4]

v1_p2 = {h: [] for h in horizons}
bv_p2 = {h: [] for h in horizons}
act2  = {h: [] for h in horizons}

for t in range(T_start, T):
    df_tr = df.iloc[:t]
    m1 = VAR(df_tr[ENDOG_OLD], exog=df_tr[EXOG]).fit(maxlags=1)
    bv = MinnesotaBayesianVar(
        endogenous=Y_all[:t], exogenous=X_all[:t], lags=LAGS,
        pi1=PI1, pi2=PI2, pi3=PI3, ar_coefficients=AR_COEF_OLD,
        iterations=300, credibility_level=0.90, verbose=False
    )
    bv.estimate()
    for h in horizons:
        if t + h > T: continue
        X_fut = X_all[t:t+h]
        fc1   = m1.forecast(df_tr[ENDOG_OLD].values[-1:], steps=h, exog_future=X_fut)
        fc_bv = bv.forecast(h=h, credibility_level=0.90, Z_p=X_fut)
        v1_p2[h].append(fc1[h-1])
        bv_p2[h].append(fc_bv[h-1, :, 0])
        act2[h].append(Y_all[t+h-1])

ratio_matrix = np.zeros((len(ENDOG_OLD), 3))
for j, h in enumerate(horizons):
    for i in range(len(ENDOG_OLD)):
        a   = np.array(act2[h])[:, i]
        rv1 = np.sqrt(np.nanmean((np.array(v1_p2[h])[:, i] - a)**2))
        rbv = np.sqrt(np.nanmean((np.array(bv_p2[h])[:, i] - a)**2))
        ratio_matrix[i, j] = rbv / rv1

fig, ax = plt.subplots(figsize=(6, 5)); fig.patch.set_facecolor('white')
cmap = mcolors.LinearSegmentedColormap.from_list('bvar', ['#27ae60', '#f5f5f5', '#c0392b'])
im   = ax.imshow(ratio_matrix, cmap=cmap, vmin=0.75, vmax=1.10, aspect='auto')
ax.set_xticks(range(3)); ax.set_xticklabels(H_LABELS, fontsize=10)
ax.set_yticks(range(len(ENDOG_OLD))); ax.set_yticklabels(VAR_LABELS, fontsize=9)

for i in range(len(ENDOG_OLD)):
    for j in range(3):
        r = ratio_matrix[i, j]
        txt_color = 'white' if r < 0.88 or r > 1.05 else 'black'
        ax.text(j, i, f'{r:.3f}', ha='center', va='center', fontsize=9, fontweight='bold', color=txt_color)

plt.colorbar(im, ax=ax, label='RMSE(BVAR) / RMSE(VARX1)  —  green < 1 = BVAR wins', shrink=0.8)
ax.set_title('Conditional OOS RMSE: BVAR(4) vs VARX(1)\nrealized exogenous paths; BVAR wins all 18 cells', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('output/readme_oos_rmse.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/readme_oos_rmse.png')


Saved: output/readme_oos_rmse.png


In [17]:
# README Fig 4: HK property price — level (I(1)) vs QoQ model variable (I(0))

df = pd.read_csv(DATA, index_col=0, parse_dates=True)

if 'hk_property_price_idx' not in df.columns:
    print('hk_property_price_idx not in CSV — skipped')
else:
    prop = df['hk_property_price_idx'].dropna()
    qoq  = df['hk_property_price_qoq'].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4)); fig.patch.set_facecolor('white')

    ax0 = axes[0]
    ax0.fill_between(prop.index, prop.values, alpha=0.25, color='#8e44ad')
    ax0.plot(prop.index, prop.values, color='#8e44ad', lw=2)
    ax0.axvspan(pd.Timestamp('2008-07-01'), pd.Timestamp('2009-06-30'), alpha=0.10, color='gray')
    ax0.axvspan(pd.Timestamp('2019-06-01'), pd.Timestamp('2020-06-30'), alpha=0.10, color='gray')
    ax0.set_title('Property Price Index — Level (I(1))\nADF p = 0.82   KPSS rejects', fontsize=9, fontweight='bold')
    ax0.set_ylabel('Index (All Classes)', fontsize=9)
    ax0.tick_params(labelsize=8)
    ax0.spines['top'].set_visible(False); ax0.spines['right'].set_visible(False)
    ymax = prop.max()
    ax0.text(pd.Timestamp('2008-10-01'), ymax*0.92, 'GFC',      fontsize=8, color='gray', ha='center')
    ax0.text(pd.Timestamp('2019-09-01'), ymax*0.92, 'Protests', fontsize=8, color='gray', ha='center')

    ax1 = axes[1]
    bar_colors = ['#c0392b' if v < 0 else '#27ae60' for v in qoq.values]
    ax1.bar(qoq.index, qoq.values, color=bar_colors, width=60, alpha=0.85)
    ax1.axhline(0, color='k', lw=0.8)
    ax1.set_title('Property Price QoQ % — Model Variable (I(0))\nADF p = 0.000', fontsize=9, fontweight='bold')
    ax1.set_ylabel('QoQ % change', fontsize=9)
    ax1.tick_params(labelsize=8)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    fig.suptitle('HK Residential Property — Why We Transform to QoQ', fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('output/readme_property_price.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print('Saved: output/readme_property_price.png')


Saved: output/readme_property_price.png
